# Brusselator reaction–diffusion: interactive loss audit

The two periodic fields obey

$$u_t=D_u\Delta u+A-(B+1)u+u^2v,\qquad v_t=D_v\Delta v+Bu-u^2v.$$

The dashboard includes the joint spatial point cloud $\{(u(x),v(x))\}$ used by sliced Wasserstein, marginal spectra, and threshold geometry. The contour view is a topology-oriented preview rather than the final differentiable persistence loss.


In [1]:
import os
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display

case_name = "brusselator"
smoke_mode = os.environ.get("PHYCOFLOW_VIZ_SMOKE", "0") == "1"
seed = 11
n = 16 if smoke_mode else 64
domain_length = 20.0
dt = 0.01
final_time = 0.04 if smoke_mode else 12.0
save_every = 1 if smoke_mode else 5
A, B = 1.0, 3.0
diffusivity_u, diffusivity_v = 1.0, 0.1
rng = np.random.default_rng(seed)


In [2]:
x = np.linspace(0.0, domain_length, n, endpoint=False)
y = np.linspace(0.0, domain_length, n, endpoint=False)
dx = domain_length / n
wave = 2.0 * np.pi * np.fft.fftfreq(n, d=dx)
kx, ky = np.meshgrid(wave, wave, indexing="xy")
k2 = kx**2 + ky**2
diffuse_u = np.exp(-0.5 * dt * diffusivity_u * k2)
diffuse_v = np.exp(-0.5 * dt * diffusivity_v * k2)

def smooth_noise(scale=0.06):
    raw_hat = np.fft.fft2(rng.standard_normal((n, n)))
    filt = np.exp(-0.12 * k2**2)
    field = np.fft.ifft2(raw_hat * filt).real
    field -= field.mean()
    return scale * field / field.std()

def reaction(u, v):
    coupling = u**2 * v
    return A - (B + 1.0) * u + coupling, B * u - coupling

def reaction_rk4(u, v):
    k1u, k1v = reaction(u, v)
    k2u, k2v = reaction(u + 0.5 * dt * k1u, v + 0.5 * dt * k1v)
    k3u, k3v = reaction(u + 0.5 * dt * k2u, v + 0.5 * dt * k2v)
    k4u, k4v = reaction(u + dt * k3u, v + dt * k3v)
    return (u + dt * (k1u + 2.0 * k2u + 2.0 * k3u + k4u) / 6.0,
            v + dt * (k1v + 2.0 * k2v + 2.0 * k3v + k4v) / 6.0)

u = A + smooth_noise()
v = B / A + smooth_noise()
snapshots = [np.stack((u, v))]
times_list = [0.0]
n_steps = int(round(final_time / dt))
for step in range(1, n_steps + 1):
    u = np.fft.ifft2(diffuse_u * np.fft.fft2(u)).real
    v = np.fft.ifft2(diffuse_v * np.fft.fft2(v)).real
    u, v = reaction_rk4(u, v)
    u = np.fft.ifft2(diffuse_u * np.fft.fft2(u)).real
    v = np.fft.ifft2(diffuse_v * np.fft.fft2(v)).real
    if step % save_every == 0 or step == n_steps:
        snapshots.append(np.stack((u, v)))
        times_list.append(step * dt)

times = np.asarray(times_list)
fields = np.asarray(snapshots)  # [time, channel=(u,v), y, x]


In [3]:
def laplacian(field):
    return np.fft.ifft2(-k2 * np.fft.fft2(field)).real

residual_u, residual_v, time_u, time_v = [], [], [], []
for index in range(1, times.size - 1):
    u_mid, v_mid = fields[index]
    ut = (fields[index + 1, 0] - fields[index - 1, 0]) / (times[index + 1] - times[index - 1])
    vt = (fields[index + 1, 1] - fields[index - 1, 1]) / (times[index + 1] - times[index - 1])
    ru, rv = reaction(u_mid, v_mid)
    residual_u.append(ut - diffusivity_u * laplacian(u_mid) - ru)
    residual_v.append(vt - diffusivity_v * laplacian(v_mid) - rv)
    time_u.append(ut)
    time_v.append(vt)
diagnostics = {
    "relative_pde_residual_u": float(np.linalg.norm(residual_u) / (np.linalg.norm(time_u) + 1.0e-12)),
    "relative_pde_residual_v": float(np.linalg.norm(residual_v) / (np.linalg.norm(time_v) + 1.0e-12)),
    "finite": bool(np.isfinite(fields).all()),
    "minimum_concentration": float(fields.min()),
    "maximum_concentration": float(fields.max()),
}
diagnostics


{'relative_pde_residual_u': 0.014776876683395626,
 'relative_pde_residual_v': 0.011999068447204185,
 'finite': True,
 'minimum_concentration': 0.3716930461999898,
 'maximum_concentration': 4.5975109800315295}

In [4]:
modes = np.fft.fftfreq(n) * n
mx, my = np.meshgrid(modes, modes, indexing="xy")
radius_index = np.sqrt(mx**2 + my**2).astype(int)

def radial_spectrum(field):
    power = np.abs(np.fft.fft2(field - field.mean()))**2 / field.size**2
    total = np.bincount(radius_index.ravel(), weights=power.ravel())
    count = np.maximum(np.bincount(radius_index.ravel()), 1)
    return np.arange(total.size), total / count

def draw_frame(frame_index, axes):
    for axis in axes.ravel():
        axis.clear()
    u_frame, v_frame = fields[frame_index]
    axes[0, 0].imshow(u_frame, origin="lower", cmap="magma")
    axes[0, 0].set_title(f"u, t={times[frame_index]:.2f}")
    axes[0, 1].imshow(v_frame, origin="lower", cmap="viridis")
    axes[0, 1].set_title(f"v, t={times[frame_index]:.2f}")
    stride = max(1, u_frame.size // 2000)
    axes[0, 2].scatter(u_frame.ravel()[::stride], v_frame.ravel()[::stride], s=5, alpha=0.35)
    axes[0, 2].set(xlabel="u", ylabel="v", title="Joint value cloud (SW input)")
    axes[1, 0].hist(u_frame.ravel(), bins=30, density=True, alpha=0.6, label="u")
    axes[1, 0].hist(v_frame.ravel(), bins=30, density=True, alpha=0.6, label="v")
    axes[1, 0].legend(); axes[1, 0].set_title("Marginal PDFs")
    ku, su = radial_spectrum(u_frame)
    kv, sv = radial_spectrum(v_frame)
    axes[1, 1].semilogy(ku[1:], su[1:] + 1.0e-18, label="u")
    axes[1, 1].semilogy(kv[1:], sv[1:] + 1.0e-18, label="v")
    axes[1, 1].legend(); axes[1, 1].set_title("Per-field spectra")
    axes[1, 2].contour(u_frame, levels=[np.median(u_frame)], colors="tab:red")
    axes[1, 2].contour(v_frame, levels=[np.median(v_frame)], colors="tab:blue")
    axes[1, 2].set_title("Median level-set geometry")
    return []

dashboard = None
if not smoke_mode:
    frame_indices = np.unique(np.linspace(0, times.size - 1, min(times.size, 31), dtype=int))
    figure, dashboard_axes = plt.subplots(2, 3, figsize=(12, 7), dpi=72, constrained_layout=True)
    animation = FuncAnimation(figure, lambda i: draw_frame(i, dashboard_axes), frames=frame_indices, interval=180, repeat=True)
    plt.close(figure)
    dashboard = HTML(animation.to_jshtml(default_mode="once"))
    display(dashboard)
